# 

## 

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report
import joblib

df = pd.read_csv('../data/processed/metricas_centralidade.csv')

# Labels
q66 = df['betweenness_centrality'].quantile(0.66)
q90 = df['betweenness_centrality'].quantile(0.90)

def classify(bc):
    if bc >= q90: return 2
    elif bc >= q66: return 1
    else: return 0

df['label'] = df['betweenness_centrality'].apply(classify)

# Features: without betweenness, but with coordinates
features = ['degree', 'degree_centrality', 'closeness_centrality', 'lat', 'lon']
X = df[features].values
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Class weights to compensate imbalanced classes
from sklearn.utils.class_weight import compute_sample_weight
sample_weights = compute_sample_weight('balanced', y_train)

model = GradientBoostingClassifier(
    n_estimators=200,
    max_depth=5,
    min_samples_leaf=10,
    random_state=42,
)
model.fit(X_train, y_train, sample_weight=sample_weights)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['Peripheral', 'Intermediate', 'Hub']))

scores = cross_val_score(model, X, y, cv=5, scoring='f1_macro')
print(f'F1 macro (CV): {scores.mean():.3f} +/- {scores.std():.3f}')

# Feature importance
print('\nFeature Importance:')
for name, imp in sorted(zip(features, model.feature_importances_), key=lambda x: -x[1]):
    print(f'  {name:25s} {imp:.4f}')

joblib.dump(model, 'sp_transit_classifier.joblib')
print('\nModel saved!')

              precision    recall  f1-score   support

  Peripheral       0.85      0.75      0.80      2890
Intermediate       0.44      0.48      0.46      1051
         Hub       0.41      0.64      0.50       438

    accuracy                           0.68      4379
   macro avg       0.57      0.62      0.59      4379
weighted avg       0.71      0.68      0.69      4379

F1 macro (CV): 0.426 +/- 0.049

Feature Importance:
  lat                       0.2793
  lon                       0.2604
  closeness_centrality      0.2566
  degree                    0.1061
  degree_centrality         0.0976

Model saved!


### Upload model to Hugging Face Hub

In [ ]:
from huggingface_hub import HfApi, create_repo
import json

REPO_ID = 'cintia-shinoda/sp-transit-node-classifier'

# 1. Criar repo
create_repo(repo_id=REPO_ID, exist_ok=True)
print('Repo criado')

api = HfApi()

# 2. Upload do modelo
api.upload_file(
    path_or_fileobj='sp_transit_classifier.joblib',
    path_in_repo='model.joblib',
    repo_id=REPO_ID,
)
print('Modelo uploaded')

# 3. Upload de config
config = {
    'model_type': 'GradientBoostingClassifier',
    'framework': 'scikit-learn',
    'features': ['degree', 'degree_centrality', 'closeness_centrality', 'lat', 'lon'],
    'label_map': {'0': 'Peripheral', '1': 'Intermediate', '2': 'Hub'},
    'metrics': {
        'f1_macro_test': 0.59,
        'accuracy_test': 0.68,
        'f1_macro_cv': 0.426,
    },
    'training': {
        'n_estimators': 200,
        'max_depth': 5,
        'min_samples_leaf': 10,
        'class_balancing': 'sample_weight_balanced',
        'test_size': 0.2,
    },
    'feature_importance': {
        'lat': 0.2793,
        'lon': 0.2604,
        'closeness_centrality': 0.2566,
        'degree': 0.1061,
        'degree_centrality': 0.0976,
    },
}

with open('config.json', 'w') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

api.upload_file(
    path_or_fileobj='config.json',
    path_in_repo='config.json',
    repo_id=REPO_ID,
)
print('Config uploaded')
print(f'\nhttps://huggingface.co/{REPO_ID}')

In [2]:
from huggingface_hub import HfApi
api = HfApi()
api.create_repo(
    repo_id='cintia-shinoda/sp-transit-explorer',
    repo_type='space',
    space_sdk='gradio',
    exist_ok=True,
)
print('Space criado!')

Space criado!


In [3]:
from huggingface_hub import HfApi
api = HfApi()

api.upload_file(
    path_or_fileobj='/tmp/app.py',
    path_in_repo='app.py',
    repo_id='cintia-shinoda/sp-transit-explorer',
    repo_type='space',
)
print('app.py uploaded')
print('https://huggingface.co/spaces/cintia-shinoda/sp-transit-explorer')

app.py uploaded
https://huggingface.co/spaces/cintia-shinoda/sp-transit-explorer
